In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# join assumptions
#  TMG file: one row per rebate deal from trading server (deal, login, date, amount, jurisdiction, name)

# Transaction Summary: one row per CRM fund movement, including all deals
#   joined to TMG by (deal = trading_server_transaction_no, login = trading_server_login)

#  Accounting Equity: snapshot as of a single reporting date (dt_report = 26/11/2025)
#   joined by (jurisdiction, login, trading_server_currency = currency)
#
# Assumptions:
# - Each rebate in TMG maps to exactly one CRM transaction by (deal, login).
# - Accounting is not matched by transaction date (dt_report is a single snapshot date),
#   only by (jurisdiction, login, currency) to confirm the account exists in accounting.

In [3]:
tmg = pd.read_excel('TMG_2025-11-25_Transactions.xlsx')
transaction = pd.read_csv('Transaction Summary Report.csv')
accounting = pd.read_csv('Accounting Equity Report.csv')

In [4]:
tmg.info

<bound method DataFrame.info of             deal     login                date  \
0       60667837    809893 2025-11-25 02:02:59   
1       60667838    749057 2025-11-25 02:02:59   
2       60667839    807967 2025-11-25 02:03:00   
3       60667840    816270 2025-11-25 02:03:05   
4       60667841    824366 2025-11-25 02:03:06   
...          ...       ...                 ...   
45312  110401659  50092183 2025-11-25 18:13:06   
45313  110401673  50092183 2025-11-25 18:13:08   
45314  110433433  50068042 2025-11-25 19:17:14   
45315  110436968  50068042 2025-11-25 19:24:38   
45316  110489519  50146233 2025-11-25 23:11:21   

                              comment  amount           server  \
0      Rebate IB 2025 Daily 2111-2311    0.13  tmgm_mt4_live01   
1      Rebate IB 2025 Daily 2111-2311    2.87  tmgm_mt4_live01   
2      Rebate IB 2025 Daily 2111-2311    0.70  tmgm_mt4_live01   
3      Rebate IB 2025 Daily 2111-2311    0.80  tmgm_mt4_live01   
4      Rebate IB 2025 Daily 2111-2311

In [5]:
transaction.info

<bound method DataFrame.info of                       date  user_id   crm_transaction_no  \
0      2025-11-25 12:29:47   804135   CINT95057960901181   
1      2025-11-25 12:29:29   805302   CINT55550363065658   
2      2025-11-25 12:29:29   805302   CINT55550363065658   
3      2025-11-25 12:29:11   288660   CINT14250384753566   
4      2025-11-25 12:29:10   288660   CINT14250384753566   
...                    ...      ...                  ...   
45312  2025-11-25 08:00:33   561865   CBWC70977701157007   
45313  2025-11-25 08:00:05   236709  CUSDT63489350017780   
45314  2025-11-25 07:59:19   481565   CBWC29946265065391   
45315  2025-11-25 07:59:06   590822   CBWC92417825255558   
45316  2025-11-25 07:56:49   801909  CUSDT35746616027900   

       trading_server_transaction_no trading_server_type  \
0                          110138958                 MT5   
1                          110138894                 MT5   
2                          110138896                 MT5   
3      

In [6]:
accounting.info

<bound method DataFrame.info of          dt_report report_jurisdiction     login currency
0       26/11/2025              VFSCCN   7172491      USD
1       26/11/2025              VFSCCN   7172490      USD
2       26/11/2025              VFSCCN   7172489      USD
3       26/11/2025              VFSCCN   7172486      USD
4       26/11/2025              VFSCCN   7172485      USD
...            ...                 ...       ...      ...
499069  26/11/2025              VFSCCN  11051407      USD
499070  26/11/2025             VFSCSEA  10529961      USD
499071  26/11/2025              VFSCGC  10529958      USD
499072  26/11/2025             VFSCSEA  10529957      USD
499073  26/11/2025              VFSCGC  10529956      USD

[499074 rows x 4 columns]>

In [7]:
# Primary key: deal
# foreign key: login, jurisdiction
tmg.head(10)

,deal,login,date,comment,amount,server,name,jurisdiction,dw_check
0,60667837,809893,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,0.13,tmgm_mt4_live01,Xtaoli Myyv ta,VFSCCN,0
1,60667838,749057,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,2.87,tmgm_mt4_live01,Kuybbo Wu ta,VFSCCN,0
2,60667839,807967,2025-11-25 02:03:00,Rebate IB 2025 Daily 2111-2311,0.70,tmgm_mt4_live01,Chbofu ahyyv ta,VFSCCN,0
3,60667840,816270,2025-11-25 02:03:05,Rebate IB 2025 Daily 2111-2311,0.80,tmgm_mt4_live01,Yiahu ahbyv ta,VFSCCN,0
4,60667841,824366,2025-11-25 02:03:06,Rebate IB 2025 Daily 2111-2311,0.40,tmgm_mt4_live01,Hsiu Fyyv Yyh,VFSCGC,0
5,60667842,814282,2025-11-25 02:03:11,Rebate IB 2025 Daily 2111-2311,0.48,tmgm_mt4_live01,Xtaowyi Lv ta,VFSCCN,0
6,60667843,748690,2025-11-25 02:03:12,Rebate IB 2025 Daily 2111-2311,1.09,tmgm_mt4_live01,Fyi Chyy ta,VFSCCN,0
7,60667844,724565,2025-11-25 02:03:16,Rebate IB 2025 Daily 2111-2311,32.36,tmgm_mt4_live01,Shyyv oiyv ta,VFSCCN,0
8,60667845,805508,2025-11-25 02:03:22,Rebate IB 2025 Daily 2111-2311,17.20,tmgm_mt4_live01,Xtaoli Myyv ta,VFSCCN,0
9,60667846,785650,2025-11-25 02:03:25,Rebate IB 2025 Daily 2111-2311,434.65,tmgm_mt4_live01,ahtayyv li ta,VFSCCN,0


In [8]:
# Primary key: crm_transaction_no, 
# foreign key: trading_server_login, jurisdiction
transaction.head(10)

,date,user_id,crm_transaction_no,trading_server_transaction_no,trading_server_type,trading_server,trading_server_login,fund_amount,trading_server_currency,payment_amount,payment_currency,exchange_rate,surcharge_amount,comment,accounting_comment,jurisdiction,payment_channel
0,2025-11-25 12:29:47,804135,CINT95057960901181,110138958,MT5,TradeMaxGlobal-Live,50148899,600.0,USD,600.0,NaN,1.0,0,INT TF FM 8180438,Internal_Transfer,VFSCGC,NaN
1,2025-11-25 12:29:29,805302,CINT55550363065658,110138894,MT5,TradeMaxGlobal-Live,50149606,-100.0,USD,-100.0,NaN,1.0,0,INT TF to 50150980,Internal_Transfer,VFSCCN,NaN
2,2025-11-25 12:29:29,805302,CINT55550363065658,110138896,MT5,TradeMaxGlobal-Live,50150980,100.0,USD,100.0,NaN,1.0,0,INT TF FM 50149606,Internal_Transfer,VFSCCN,NaN
3,2025-11-25 12:29:11,288660,CINT14250384753566,132459679,MT4,TradeMaxGlobal-Live3,547524,30.0,USD,30.0,NaN,1.0,0,INT TF FM 3306301,Internal_Transfer,VFSCCN,NaN
4,2025-11-25 12:29:10,288660,CINT14250384753566,132459678,MT4,TradeMaxGlobal-Live3,3306301,-30.0,USD,-30.0,NaN,1.0,0,INT TF to 547524,Internal_Transfer,VFSCCN,NaN
5,2025-11-25 12:28:55,735643,CINT32478389286984,110138632,MT5,TradeMaxGlobal-Live,50109009,-10.0,USD,-10.0,NaN,1.0,0,INT TF to 50109008,Internal_Transfer,VFSCCN,NaN
6,2025-11-25 12:28:55,735643,CINT32478389286984,110138633,MT5,TradeMaxGlobal-Live,50109008,10.0,USD,10.0,NaN,1.0,0,INT TF FM 50109009,Internal_Transfer,VFSCCN,NaN
7,2025-11-25 12:28:54,807850,CINT95151412196889,24804846,MT4,TradeMaxGlobal-Live12,12064254,30.0,USD,30.0,NaN,1.0,0,INT TF FM 12064261,Internal_Transfer,VFSCCN,NaN
8,2025-11-25 12:28:53,807850,CINT95151412196889,24804845,MT4,TradeMaxGlobal-Live12,12064261,-30.0,USD,-30.0,NaN,1.0,0,INT TF to 12064254,Internal_Transfer,VFSCCN,NaN
9,2025-11-25 12:28:52,761193,CINT81579923200590,30195918,MT4,TradeMaxGlobal-Live11,11071324,60.0,USD,60.0,NaN,1.0,0,INT TF FM 594297,Internal_Transfer,VFSCCN,NaN


In [9]:
# Primary key: dt_report, login, currency, report_jurisdiction) as a composite primary key
# foreign key: report_jurisdiction,login,currency
accounting.head(10)

,dt_report,report_jurisdiction,login,currency
0,26/11/2025,VFSCCN,7172491,USD
1,26/11/2025,VFSCCN,7172490,USD
2,26/11/2025,VFSCCN,7172489,USD
3,26/11/2025,VFSCCN,7172486,USD
4,26/11/2025,VFSCCN,7172485,USD
5,26/11/2025,VFSCCN,7172484,USD
6,26/11/2025,VFSCCN,7172478,USD
7,26/11/2025,VFSCCN,7172476,USD
8,26/11/2025,VFSCCN,7172474,USD
9,26/11/2025,VFSCCN,7172473,USD


In [10]:
# I wanted to make sure if these two login are the same so they can merge together
tmg1 = tmg.copy()
tmg1['login'] = tmg['login'].astype(str)
transaction1 = transaction.copy()

transaction1['trading_server_login'] = transaction['trading_server_login'].astype(str)

common_logins = set(tmg1['login']) & set(transaction1['trading_server_login'])
len(common_logins)

25999

In [11]:
# Check for null values for datasets
tmg.isnull().sum()

deal            0
login           0
date            0
comment         0
amount          0
server          0
name            0
jurisdiction    0
dw_check        0
dtype: int64

In [12]:
# It appears only the transcation data has null values, in stead of simply dropping all of it
# Only non-key, optional fields (crm_transaction_no, payment_currency, payment_channel) contain null values.
# Since they are not used as join keys ，I keep the rows and leave these fields as null, to avoid losing valid transactions.

transaction.isnull().sum()

date                                 0
user_id                              0
crm_transaction_no                1333
trading_server_transaction_no        0
trading_server_type                  0
trading_server                       0
trading_server_login                 0
fund_amount                          0
trading_server_currency              0
payment_amount                       0
payment_currency                 36548
exchange_rate                        0
surcharge_amount                     0
comment                              0
accounting_comment                   0
jurisdiction                         0
payment_channel                  38061
dtype: int64

In [13]:
accounting.isnull().sum()

dt_report              0
report_jurisdiction    0
login                  0
currency               0
dtype: int64

In [14]:
# Merge, I used copy() for the 3 datasets in case I messed up

In [15]:
merge1 = pd.merge(tmg1,transaction1,how='left', 
                  left_on = ['deal','login'], 
                  right_on =['trading_server_transaction_no','trading_server_login'],
                  suffixes = ('','_tran')
                )

In [16]:
merge1.shape

(45317, 26)

In [17]:
merge1

,deal,login,date,comment,amount,server,name,jurisdiction,dw_check,date_tran,...,fund_amount,trading_server_currency,payment_amount,payment_currency,exchange_rate,surcharge_amount,comment_tran,accounting_comment,jurisdiction_tran,payment_channel
0,60667837,809893,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,0.13,tmgm_mt4_live01,Xtaoli Myyv ta,VFSCCN,0,2025-11-25 02:02:59,...,0.13,USD,0.13,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN
1,60667838,749057,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,2.87,tmgm_mt4_live01,Kuybbo Wu ta,VFSCCN,0,2025-11-25 02:02:59,...,2.87,USD,2.87,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN
2,60667839,807967,2025-11-25 02:03:00,Rebate IB 2025 Daily 2111-2311,0.70,tmgm_mt4_live01,Chbofu ahyyv ta,VFSCCN,0,2025-11-25 02:03:00,...,0.70,USD,0.70,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN
3,60667840,816270,2025-11-25 02:03:05,Rebate IB 2025 Daily 2111-2311,0.80,tmgm_mt4_live01,Yiahu ahbyv ta,VFSCCN,0,2025-11-25 02:03:05,...,0.80,USD,0.80,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN
4,60667841,824366,2025-11-25 02:03:06,Rebate IB 2025 Daily 2111-2311,0.40,tmgm_mt4_live01,Hsiu Fyyv Yyh,VFSCGC,0,2025-11-25 02:03:06,...,0.40,USD,0.40,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCGC,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45312,110401659,50092183,2025-11-25 18:13:06,Split Signal REG 12062784,-15.00,tmgm_mt5_live01,tiyvtiyv wbyv,VFSCCN,0,2025-11-25 18:13:06,...,-15.00,USD,0.00,USD,0.0,0,Split Signal REG 12062784,Split,VFSCCN,NaN
45313,110401673,50092183,2025-11-25 18:13:08,Split Signal MGMT 12062784,-20.00,tmgm_mt5_live01,tiyvtiyv wbyv,VFSCCN,0,2025-11-25 18:13:08,...,-20.00,USD,0.00,USD,0.0,0,Split Signal MGMT 12062784,Split,VFSCCN,NaN
45314,110433433,50068042,2025-11-25 19:17:14,Split Signal REG 8173312,-1.00,tmgm_mt5_live01,biyrry ojymbsoy Chyry,VFSC,0,2025-11-25 19:17:14,...,-1.00,CAD,0.00,USD,0.0,0,Split Signal REG 8173312,Split,VFSC,NaN
45315,110436968,50068042,2025-11-25 19:24:38,Split Signal MGMT 8173312,-1.00,tmgm_mt5_live01,biyrry ojymbsoy Chyry,VFSC,0,2025-11-25 19:24:38,...,-1.00,CAD,0.00,USD,0.0,0,Split Signal MGMT 8173312,Split,VFSC,NaN


In [18]:
# The format of dt_report seems dismatched
accounting1 = accounting.copy()
accounting1

,dt_report,report_jurisdiction,login,currency
0,26/11/2025,VFSCCN,7172491,USD
1,26/11/2025,VFSCCN,7172490,USD
2,26/11/2025,VFSCCN,7172489,USD
3,26/11/2025,VFSCCN,7172486,USD
4,26/11/2025,VFSCCN,7172485,USD
...,...,...,...,...
499069,26/11/2025,VFSCCN,11051407,USD
499070,26/11/2025,VFSCSEA,10529961,USD
499071,26/11/2025,VFSCGC,10529958,USD
499072,26/11/2025,VFSCSEA,10529957,USD


In [19]:
# Fixed
accounting1['dt_report'] = pd.to_datetime(accounting1['dt_report'],format = '%d/%m/%Y').dt.date
accounting1

,dt_report,report_jurisdiction,login,currency
0,2025-11-26,VFSCCN,7172491,USD
1,2025-11-26,VFSCCN,7172490,USD
2,2025-11-26,VFSCCN,7172489,USD
3,2025-11-26,VFSCCN,7172486,USD
4,2025-11-26,VFSCCN,7172485,USD
...,...,...,...,...
499069,2025-11-26,VFSCCN,11051407,USD
499070,2025-11-26,VFSCSEA,10529961,USD
499071,2025-11-26,VFSCGC,10529958,USD
499072,2025-11-26,VFSCSEA,10529957,USD


In [20]:
accounting1['login'] = accounting1['login'].astype(str)
merge2 = pd.merge(merge1,accounting1,
                  how = 'left', 
                  left_on = ['jurisdiction','login','trading_server_currency'],
                  right_on = ['report_jurisdiction','login','currency'],
                  suffixes =('','_acc') )
merge2

,deal,login,date,comment,amount,server,name,jurisdiction,dw_check,date_tran,...,payment_currency,exchange_rate,surcharge_amount,comment_tran,accounting_comment,jurisdiction_tran,payment_channel,dt_report,report_jurisdiction,currency
0,60667837,809893,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,0.13,tmgm_mt4_live01,Xtaoli Myyv ta,VFSCCN,0,2025-11-25 02:02:59,...,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN,2025-11-26,VFSCCN,USD
1,60667838,749057,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,2.87,tmgm_mt4_live01,Kuybbo Wu ta,VFSCCN,0,2025-11-25 02:02:59,...,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN,2025-11-26,VFSCCN,USD
2,60667839,807967,2025-11-25 02:03:00,Rebate IB 2025 Daily 2111-2311,0.70,tmgm_mt4_live01,Chbofu ahyyv ta,VFSCCN,0,2025-11-25 02:03:00,...,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN,2025-11-26,VFSCCN,USD
3,60667840,816270,2025-11-25 02:03:05,Rebate IB 2025 Daily 2111-2311,0.80,tmgm_mt4_live01,Yiahu ahbyv ta,VFSCCN,0,2025-11-25 02:03:05,...,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCCN,NaN,2025-11-26,VFSCCN,USD
4,60667841,824366,2025-11-25 02:03:06,Rebate IB 2025 Daily 2111-2311,0.40,tmgm_mt4_live01,Hsiu Fyyv Yyh,VFSCGC,0,2025-11-25 02:03:06,...,NaN,1.0,0,Rebate IB 2025 Daily 2111-2311,Rebate,VFSCGC,NaN,2025-11-26,VFSCGC,USD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45312,110401659,50092183,2025-11-25 18:13:06,Split Signal REG 12062784,-15.00,tmgm_mt5_live01,tiyvtiyv wbyv,VFSCCN,0,2025-11-25 18:13:06,...,USD,0.0,0,Split Signal REG 12062784,Split,VFSCCN,NaN,2025-11-26,VFSCCN,USD
45313,110401673,50092183,2025-11-25 18:13:08,Split Signal MGMT 12062784,-20.00,tmgm_mt5_live01,tiyvtiyv wbyv,VFSCCN,0,2025-11-25 18:13:08,...,USD,0.0,0,Split Signal MGMT 12062784,Split,VFSCCN,NaN,2025-11-26,VFSCCN,USD
45314,110433433,50068042,2025-11-25 19:17:14,Split Signal REG 8173312,-1.00,tmgm_mt5_live01,biyrry ojymbsoy Chyry,VFSC,0,2025-11-25 19:17:14,...,USD,0.0,0,Split Signal REG 8173312,Split,VFSC,NaN,2025-11-26,VFSC,CAD
45315,110436968,50068042,2025-11-25 19:24:38,Split Signal MGMT 8173312,-1.00,tmgm_mt5_live01,biyrry ojymbsoy Chyry,VFSC,0,2025-11-25 19:24:38,...,USD,0.0,0,Split Signal MGMT 8173312,Split,VFSC,NaN,2025-11-26,VFSC,CAD


In [21]:
merge2.columns

Index(['deal', 'login', 'date', 'comment', 'amount', 'server', 'name',
       'jurisdiction', 'dw_check', 'date_tran', 'user_id',
       'crm_transaction_no', 'trading_server_transaction_no',
       'trading_server_type', 'trading_server', 'trading_server_login',
       'fund_amount', 'trading_server_currency', 'payment_amount',
       'payment_currency', 'exchange_rate', 'surcharge_amount', 'comment_tran',
       'accounting_comment', 'jurisdiction_tran', 'payment_channel',
       'dt_report', 'report_jurisdiction', 'currency'],
      dtype='object')

In [22]:
merge1.columns

Index(['deal', 'login', 'date', 'comment', 'amount', 'server', 'name',
       'jurisdiction', 'dw_check', 'date_tran', 'user_id',
       'crm_transaction_no', 'trading_server_transaction_no',
       'trading_server_type', 'trading_server', 'trading_server_login',
       'fund_amount', 'trading_server_currency', 'payment_amount',
       'payment_currency', 'exchange_rate', 'surcharge_amount', 'comment_tran',
       'accounting_comment', 'jurisdiction_tran', 'payment_channel'],
      dtype='object')

In [23]:
# Checking if I merged correctly
merge2[['dt_report', 'report_jurisdiction', 'currency']].isna().all()

dt_report              False
report_jurisdiction    False
currency               False
dtype: bool

In [24]:
# rearranging columns

In [25]:
# Assumption: use `fund_amount` from Transaction Summary as the final Amount,
# because it represents the signed transaction value. TMG `amount` is equivalent.
df = merge2.copy()
cols = [
    'deal',
    'login',
    'name',
    'date',
    'comment_tran',
    'fund_amount',
    'currency',
    'jurisdiction',
    'accounting_comment',
    'date'
]         
df = df[cols].copy()
df.head()

,deal,login,name,date,comment_tran,fund_amount,currency,jurisdiction,accounting_comment,date
0,60667837,809893,Xtaoli Myyv ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,0.13,USD,VFSCCN,Rebate,2025-11-25 02:02:59
1,60667838,749057,Kuybbo Wu ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,2.87,USD,VFSCCN,Rebate,2025-11-25 02:02:59
2,60667839,807967,Chbofu ahyyv ta,2025-11-25 02:03:00,Rebate IB 2025 Daily 2111-2311,0.70,USD,VFSCCN,Rebate,2025-11-25 02:03:00
3,60667840,816270,Yiahu ahbyv ta,2025-11-25 02:03:05,Rebate IB 2025 Daily 2111-2311,0.80,USD,VFSCCN,Rebate,2025-11-25 02:03:05
4,60667841,824366,Hsiu Fyyv Yyh,2025-11-25 02:03:06,Rebate IB 2025 Daily 2111-2311,0.40,USD,VFSCGC,Rebate,2025-11-25 02:03:06


In [26]:
df.isnull().sum

<bound method DataFrame.sum of         deal  login   name   date  comment_tran  fund_amount  currency  \
0      False  False  False  False         False        False     False   
1      False  False  False  False         False        False     False   
2      False  False  False  False         False        False     False   
3      False  False  False  False         False        False     False   
4      False  False  False  False         False        False     False   
...      ...    ...    ...    ...           ...          ...       ...   
45312  False  False  False  False         False        False     False   
45313  False  False  False  False         False        False     False   
45314  False  False  False  False         False        False     False   
45315  False  False  False  False         False        False     False   
45316  False  False  False  False         False        False     False   

       jurisdiction  accounting_comment   date  
0             False            

In [27]:
# helper function

In [28]:
df = df.rename(columns={'comment_tran': 'comment','fund_amount' : 'Amount'})
df

,deal,login,name,date,comment,Amount,currency,jurisdiction,accounting_comment,date
0,60667837,809893,Xtaoli Myyv ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,0.13,USD,VFSCCN,Rebate,2025-11-25 02:02:59
1,60667838,749057,Kuybbo Wu ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,2.87,USD,VFSCCN,Rebate,2025-11-25 02:02:59
2,60667839,807967,Chbofu ahyyv ta,2025-11-25 02:03:00,Rebate IB 2025 Daily 2111-2311,0.70,USD,VFSCCN,Rebate,2025-11-25 02:03:00
3,60667840,816270,Yiahu ahbyv ta,2025-11-25 02:03:05,Rebate IB 2025 Daily 2111-2311,0.80,USD,VFSCCN,Rebate,2025-11-25 02:03:05
4,60667841,824366,Hsiu Fyyv Yyh,2025-11-25 02:03:06,Rebate IB 2025 Daily 2111-2311,0.40,USD,VFSCGC,Rebate,2025-11-25 02:03:06
...,...,...,...,...,...,...,...,...,...,...
45312,110401659,50092183,tiyvtiyv wbyv,2025-11-25 18:13:06,Split Signal REG 12062784,-15.00,USD,VFSCCN,Split,2025-11-25 18:13:06
45313,110401673,50092183,tiyvtiyv wbyv,2025-11-25 18:13:08,Split Signal MGMT 12062784,-20.00,USD,VFSCCN,Split,2025-11-25 18:13:08
45314,110433433,50068042,biyrry ojymbsoy Chyry,2025-11-25 19:17:14,Split Signal REG 8173312,-1.00,CAD,VFSC,Split,2025-11-25 19:17:14
45315,110436968,50068042,biyrry ojymbsoy Chyry,2025-11-25 19:24:38,Split Signal MGMT 8173312,-1.00,CAD,VFSC,Split,2025-11-25 19:24:38


In [29]:
%run code_snippet.ipynb

In [30]:
df['Comment Format'] = df.apply(
    lambda x: comment_format_TMG(x['accounting_comment'].upper(),
                                 x['comment'].upper(),
                                 x['jurisdiction']).upper(),
    axis=1
)

In [31]:
df['Accounting Comments'] = (
    df['login'].astype(str)
    + '+' 
    + df['currency'].astype(str)
    + '+' + df['jurisdiction'].astype(str)   
    + '+' 
    + df['Comment Format'].fillna('')
)
df.columns = df.columns.str.capitalize()
df

,Deal,Login,Name,Date,Comment,Amount,Currency,Jurisdiction,Accounting_comment,Date,Comment format,Accounting comments
0,60667837,809893,Xtaoli Myyv ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,0.13,USD,VFSCCN,Rebate,2025-11-25 02:02:59,REBATE TF + TD +,809893+USD+VFSCCN+REBATE TF + TD +
1,60667838,749057,Kuybbo Wu ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,2.87,USD,VFSCCN,Rebate,2025-11-25 02:02:59,REBATE TF + TD +,749057+USD+VFSCCN+REBATE TF + TD +
2,60667839,807967,Chbofu ahyyv ta,2025-11-25 02:03:00,Rebate IB 2025 Daily 2111-2311,0.70,USD,VFSCCN,Rebate,2025-11-25 02:03:00,REBATE TF + TD +,807967+USD+VFSCCN+REBATE TF + TD +
3,60667840,816270,Yiahu ahbyv ta,2025-11-25 02:03:05,Rebate IB 2025 Daily 2111-2311,0.80,USD,VFSCCN,Rebate,2025-11-25 02:03:05,REBATE TF + TD +,816270+USD+VFSCCN+REBATE TF + TD +
4,60667841,824366,Hsiu Fyyv Yyh,2025-11-25 02:03:06,Rebate IB 2025 Daily 2111-2311,0.40,USD,VFSCGC,Rebate,2025-11-25 02:03:06,REBATE TF + TD +,824366+USD+VFSCGC+REBATE TF + TD +
...,...,...,...,...,...,...,...,...,...,...,...,...
45312,110401659,50092183,tiyvtiyv wbyv,2025-11-25 18:13:06,Split Signal REG 12062784,-15.00,USD,VFSCCN,Split,2025-11-25 18:13:06,SPLIT SIGNAL TF,50092183+USD+VFSCCN+SPLIT SIGNAL TF
45313,110401673,50092183,tiyvtiyv wbyv,2025-11-25 18:13:08,Split Signal MGMT 12062784,-20.00,USD,VFSCCN,Split,2025-11-25 18:13:08,SPLIT SIGNAL TF,50092183+USD+VFSCCN+SPLIT SIGNAL TF
45314,110433433,50068042,biyrry ojymbsoy Chyry,2025-11-25 19:17:14,Split Signal REG 8173312,-1.00,CAD,VFSC,Split,2025-11-25 19:17:14,SPLIT SIGNAL TF,50068042+CAD+VFSC+SPLIT SIGNAL TF
45315,110436968,50068042,biyrry ojymbsoy Chyry,2025-11-25 19:24:38,Split Signal MGMT 8173312,-1.00,CAD,VFSC,Split,2025-11-25 19:24:38,SPLIT SIGNAL TF,50068042+CAD+VFSC+SPLIT SIGNAL TF


In [36]:
df['Date'] = pd.to_datetime(df['Date'])


df['Transaction Date'] = df['Date'].dt.strftime('%d/%m/%Y')

In [39]:
col_order = [
    'Deal',
    'Login',
    'Name',
    'Date',
    'Comment',
    'Amount',
    'Currency',
    'Comment format',
    'Accounting comments',
    'Transaction Date'
]
df = df[col_order].copy()
df

,Deal,Login,Name,Date,Comment,Amount,Currency,Comment format,Accounting comments,Transaction Date
0,60667837,809893,Xtaoli Myyv ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,0.13,USD,REBATE TF + TD +,809893+USD+VFSCCN+REBATE TF + TD +,25/11/2025
1,60667838,749057,Kuybbo Wu ta,2025-11-25 02:02:59,Rebate IB 2025 Daily 2111-2311,2.87,USD,REBATE TF + TD +,749057+USD+VFSCCN+REBATE TF + TD +,25/11/2025
2,60667839,807967,Chbofu ahyyv ta,2025-11-25 02:03:00,Rebate IB 2025 Daily 2111-2311,0.70,USD,REBATE TF + TD +,807967+USD+VFSCCN+REBATE TF + TD +,25/11/2025
3,60667840,816270,Yiahu ahbyv ta,2025-11-25 02:03:05,Rebate IB 2025 Daily 2111-2311,0.80,USD,REBATE TF + TD +,816270+USD+VFSCCN+REBATE TF + TD +,25/11/2025
4,60667841,824366,Hsiu Fyyv Yyh,2025-11-25 02:03:06,Rebate IB 2025 Daily 2111-2311,0.40,USD,REBATE TF + TD +,824366+USD+VFSCGC+REBATE TF + TD +,25/11/2025
...,...,...,...,...,...,...,...,...,...,...
45312,110401659,50092183,tiyvtiyv wbyv,2025-11-25 18:13:06,Split Signal REG 12062784,-15.00,USD,SPLIT SIGNAL TF,50092183+USD+VFSCCN+SPLIT SIGNAL TF,25/11/2025
45313,110401673,50092183,tiyvtiyv wbyv,2025-11-25 18:13:08,Split Signal MGMT 12062784,-20.00,USD,SPLIT SIGNAL TF,50092183+USD+VFSCCN+SPLIT SIGNAL TF,25/11/2025
45314,110433433,50068042,biyrry ojymbsoy Chyry,2025-11-25 19:17:14,Split Signal REG 8173312,-1.00,CAD,SPLIT SIGNAL TF,50068042+CAD+VFSC+SPLIT SIGNAL TF,25/11/2025
45315,110436968,50068042,biyrry ojymbsoy Chyry,2025-11-25 19:24:38,Split Signal MGMT 8173312,-1.00,CAD,SPLIT SIGNAL TF,50068042+CAD+VFSC+SPLIT SIGNAL TF,25/11/2025


In [40]:
# sort value

In [43]:
df = df.sort_values(by=['Login','Deal'],ascending= False)
df

,Deal,Login,Name,Date,Comment,Amount,Currency,Comment format,Accounting comments,Transaction Date
21886,40098018,9172698,chbyvby xiy,2025-11-25 10:22:51,INT TF FM 7174719,5000.00,USD,INTERNAL TF,9172698+USD+VFSCCN+INTERNAL TF,25/11/2025
19999,40080168,9172692,ahihbo ybyv,2025-11-25 00:23:19,INT TF FM 12015661,1200.00,USD,INTERNAL TF,9172692+USD+VFSCCN+INTERNAL TF,25/11/2025
21946,40099418,9172670,kbyvyu liy,2025-11-25 10:48:27,INT TF to 12062607,-926.14,USD,INTERNAL TF,9172670+USD+VFSCCN+INTERNAL TF,25/11/2025
21470,40090108,9172521,yuy you,2025-11-25 05:57:21,INT TF to 11071764,-100.00,USD,INTERNAL TF,9172521+USD+VFSCCN+INTERNAL TF,25/11/2025
21430,40089259,9172521,yuy you,2025-11-25 05:25:24,INT TF FM 9172257,100.00,USD,INTERNAL TF,9172521+USD+VFSCCN+INTERNAL TF,25/11/2025
...,...,...,...,...,...,...,...,...,...,...
23418,19227355,10500206,Wbi Chuy Lbm,2025-11-25 02:52:00,Rebate IB 2025 Daily 2111-2311,3.57,USD,REBATE TF + TD +,10500206+USD+VFSCGC+REBATE TF + TD +,25/11/2025
23664,19232611,10500058,boryboyv booysryy,2025-11-25 10:16:34,INT TF to 50152090,-100.00,USD,INTERNAL TF,10500058+USD+VFSCSEA+INTERNAL TF,25/11/2025
23662,19232505,10500058,boryboyv booysryy,2025-11-25 10:07:39,Deposit Via ASIA 1st,100.00,USD,NULL,10500058+USD+VFSCSEA+NULL,25/11/2025
23367,19227295,10500050,Vby HObyv yvUYyy,2025-11-25 02:46:56,Rebate IB 2025 Daily 2111-2311,109.75,USD,REBATE TF + TD +,10500050+USD+VFSCSEA+REBATE TF + TD +,25/11/2025


In [45]:
# extract xlsx file
output_file = "VFSC_by_TonyLan.xlsx"

with pd.ExcelWriter(output_file) as writer:
    
    for cur, sub in df.groupby("Currency"):
        
        sheet_name = str(cur)[:10] if pd.notna(cur) else "Unknown"
        sub.to_excel(writer, sheet_name=sheet_name, index=False)

print("Finished：", output_file)

Finished： VFSC_by_TonyLan.xlsx
